# Optimization Campaign (HITL)

<details>
<summary>About this notebook</summary>

Interactive notebook for running prompt optimization campaigns with full human-in-the-loop control.

**Workflow:** Config → Load Data → Diagnostics → Baseline Eval → Optimization Round → LLM Suggestions → Repeat

**Pipeline:** The pipeline config is imported from the TermNorm backend and can be edited in the Pipeline Config cell below. By default, the LLM reranking step is included — remove it to run token-match-only evaluation.

**Prerequisites:**
1. **TermNorm backend running** at `http://127.0.0.1:8000` — required for syncing experiment data and for all evaluation steps.
2. **Groq API key** set in `.env`.
3. After the first sync, **restart the kernel** once so the setup cell loads the freshly synced data.

</details>

## 1. Setup & Load Data

<details>
<summary>Expected first-run output</summary>

On first run (or after clearing stored data), you'll see the auto-sync message — this is expected:
```
Stored data has no traces — syncing from http://127.0.0.1:8000 ...
Experiment : production_historical
Mappings   : 887 total, 812 with verified ground truth
Queries    : 40  |  Session terms: 93
Loaded 40 eval queries
Ready.
```

</details>

In [ ]:
#@title Setup & imports
%load_ext autoreload
%autoreload 2

import json
import os

import pandas as pd

from _campaign_lib import (
    init_services, analyze_candidate_coverage,
    load_baseline_prompt,
    generate_candidates, select_round_winner, generate_suggestions,
    display_suggestions, save_campaign_winner,
    display_progress, run_optimization_loop, setup_llm,
    run_feedback_cycle_notebook,
    # Grid search
    DEFAULT_GRID_AXES,
    validate_grid_config,
    build_grid_points, run_grid_search, display_grid_results,
    select_grid_winner, analyze_grid_results, load_eval_dataset,
    resume_or_build_grid,
    # Grid plan discovery
    list_grid_plans, load_grid_plan_results, merge_grid_results,
    # Pipeline config
    load_pipeline_config, build_pipeline_params, PIPELINE_STEP_PARAMS,
    # Smart Prompt Search
    build_diagnostic_set, sensitivity_scan, adaptive_search,
    display_axis_profiles, load_variant_library,
    resume_or_build_diagnostic, select_scan_winner_notebook,
    build_historical_index, synthesize_sensitivity,
    show_scan_coverage, show_data_inventory,
    # Notebook-facing wrappers (collapsed cells)
    show_pipeline_config, show_grid_overview, run_baseline_eval,
    select_and_seed_grid_winner, run_manual_round,
)

svc = await init_services()

campaign_rounds = []
print("Ready.")

## 2. Campaign Config

<details>
<summary>Details</summary>

Edit this cell and re-run to change settings for optimization and the evaluation LLM. The pipeline config is imported from the backend in the next cell — use `pipeline_overrides` below to tweak specific pipeline parameters. After each round, the LLM suggestion cell will print a modified config you can copy back here.

</details>

In [2]:
campaign_config = {
    "queries_per_eval": 35,              # queries per optimization evaluation step
    "exploration_rate": 0.5,             # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "pipeline_overrides": {              # Override specific pipeline params (omitted = backend default):
        # "profiling_temperature": 0.3,
    },
    "optimization": {
        "n_variants": 5,
        "creativity": 0.7,
        "improvement_threshold": 0.01,
        "patience": 3,                   # rounds without improvement before auto-stop
        "max_rounds": 10,
    },
    "eval_llm": {
        "model": "meta-llama/llama-4-maverick-17b-128e-instruct",
        "provider_url": "https://api.groq.com/openai/v1/chat/completions",
        "temperature": 0,
        "max_tokens": 4000,
    },
    "grid_search": {
        "context": "A terminology normalization pipeline that matches raw material "
                   "descriptions to standardized database terms using entity profiling "
                   "and candidate ranking.",
        # OR provide structured fields directly:
        # "context_fields": {"persona": "...", "task_intent": "...", ...},
        "grid_budget": 35,            # exact budget (0=full grid)
        "eval_queries_per_point": 1,  # queries per grid point (0=use all eval_data)
        "shared_queries": False,      # False=different random queries per point
        "seed": 42,
        "top_k": 5,
        "use_defaults": True,        # use DEFAULT_GRID_AXES library
        # "custom_axes": {...},      # override specific axes
    },
    "smart_search": {
        "n_diagnostic": 6,
        "max_rounds": 3,
        "stop_threshold": 0.0,
    },
}

print(json.dumps(campaign_config, indent=2))

{
  "queries_per_eval": 35,
  "exploration_rate": 0.5,
  "pipeline_overrides": {},
  "optimization": {
    "n_variants": 5,
    "creativity": 0.7,
    "improvement_threshold": 0.01,
    "patience": 3,
    "max_rounds": 10
  },
  "eval_llm": {
    "model": "meta-llama/llama-4-maverick-17b-128e-instruct",
    "provider_url": "https://api.groq.com/openai/v1/chat/completions",
    "temperature": 0,
    "max_tokens": 4000
  },
  "grid_search": {
    "context": "A terminology normalization pipeline that matches raw material descriptions to standardized database terms using entity profiling and candidate ranking.",
    "grid_budget": 35,
    "eval_queries_per_point": 1,
    "shared_queries": false,
    "seed": 42,
    "top_k": 5,
    "use_defaults": true
  },
  "smart_search": {
    "n_diagnostic": 6,
    "max_rounds": 3,
    "stop_threshold": 0.0
  }
}


In [3]:
#@title Pipeline Config — import from backend + edit steps
pipeline_params = show_pipeline_config(svc, campaign_config)

Pipeline: TermNormPipeline (production_v1)
Notation: LLM1-TokenMatching-LLM2
Steps (3):
  1. entity_profiling (LLMGeneration)
  2. token_matching (DeterministicFunction)
  3. llm_ranking (LLMGeneration)

Active steps: entity_profiling -> token_matching -> llm_ranking
Pipeline params: {'steps': ['entity_profiling', 'token_matching', 'llm_ranking']}

Tunable parameters:
  entity_profiling:
    profiling_max_tokens               (no search values defined)
    profiling_temperature              (no search values defined)
    raw_content_limit                  (no search values defined)
  token_matching:
    max_token_candidates               search: [5, 10, 20]
    relevance_weight_core              (no search values defined)
  llm_ranking:
    ranking_max_tokens                 (no search values defined)
    ranking_prompt                     (overridden by optimizer)
    ranking_sample_size                search: [3, 5, 10]
    ranking_temperature                search: [0.0, 0.3, 0.7, 1

## 2.5 Exploration Preview

<details>
<summary>Details</summary>

Preview the grid search sampling configuration before running.

</details>

In [4]:
#@title Preview: queries_per_eval + exploration_rate
queries_per_eval = campaign_config["queries_per_eval"]
exploration_rate = campaign_config["exploration_rate"]
gs = campaign_config["grid_search"]
grid_budget = gs.get("grid_budget", 0)
eval_queries_per_point = gs.get("eval_queries_per_point", 1)
shared_queries = gs.get("shared_queries", False)

print(f"queries_per_eval       : {queries_per_eval} queries per optimization step")
print(f"exploration_rate       : {exploration_rate:.2f} (0.0=conservative, 1.0=aggressive)")
print(f"grid_budget            : {grid_budget if grid_budget > 0 else 'full grid'}")
print(f"eval_queries_per_point : {eval_queries_per_point}")
print(f"shared_queries         : {shared_queries}")
print(f"seed                   : {gs.get('seed', 42)}")
print()
if grid_budget > 0:
    print(f"Grid points to evaluate: {grid_budget}")
    if eval_queries_per_point > 0:
        print(f"Queries per grid point: {eval_queries_per_point}")
        if not shared_queries:
            print("Each grid point gets different random queries (seeded)")
else:
    print("Grid size depends on axes — will be shown after building grid points.")

queries_per_eval       : 35 queries per optimization step
exploration_rate       : 0.50 (0.0=conservative, 1.0=aggressive)
grid_budget            : 35
eval_queries_per_point : 1
shared_queries         : False
seed                   : 42

Grid points to evaluate: 35
Queries per grid point: 1
Each grid point gets different random queries (seeded)


In [5]:
#@title Improvement areas (domain expert guidance)
# Describe where you think improvement is most likely.
# This feeds into the LLM consultant in cell 4.5a to produce targeted advice.
# Leave empty to skip consultation.
improvement_areas = "profile schema quality, web search relevance"

if improvement_areas:
    print(f"Improvement areas: {improvement_areas}")
    print("These will be passed to the restructure-context LLM for strategic consultation.")
else:
    print("No improvement areas set — restructure-context will run without consultation.")

Improvement areas: profile schema quality, web search relevance
These will be passed to the restructure-context LLM for strategic consultation.


## 3. Diagnostic — Candidate Coverage

<details>
<summary>Details</summary>

For each query: is the ground truth in the token-matched candidates? At what rank? This determines whether ranking optimization is viable (ground truth must be in the candidate set for the ranker to promote it).

</details>

In [6]:
#@title Candidate coverage analysis
eval_data = load_eval_dataset(svc["store"], svc["backend_id"], svc["experiment_id"])
cov_df = analyze_candidate_coverage(eval_data)

Loaded 40 eval queries
Eval runs: 111 completed runs, 15 in-progress
  run_id                        name                 model                      temp  accuracy  queries
  baseline_816203b2             Baseline             meta-llama/llama-4-mav...  0     17.5%     40     
  grid_842a9010                 grid_combo_0         meta-llama/llama-4-mav...  0     0.0%      35     
  grid_6759eb45                 grid_combo_1         meta-llama/llama-4-mav...  0     0.0%      35     
  grid_aa8001f4                 grid_point_0                                    0.0   0.0%      1      
  grid_19f42064                 grid_point_1                                    0.0   0.0%      1      
  grid_2470cf35                 grid_point_2                                    0.0   0.0%      1      
  grid_9a91bc16                 grid_point_3                                    0.0   0.0%      1      
  grid_51346dcd                 grid_point_4                                    0.0   0.0%      1  

In [7]:
#@title Sample entity profiles (qualitative check)
n_profile_samples = 3
samples = [r for r in eval_data if r.get("pipeline_data", {}).get("entity_profile")][:n_profile_samples]

for i, s in enumerate(samples):
    profile = s["pipeline_data"]["entity_profile"]
    print(f"--- Sample {i+1}: {s['query'][:60]} ---")
    print(f"  Core concept: {profile.get('core_concept', '?')}")
    print(f"  Profile keys: {list(profile.keys())}")
    print(f"  Ground truth: {s['ground_truth']}")
    candidates = s.get("pipeline_data", {}).get("token_matched_candidates", [])[:5]
    print(f"  Top 5 candidates: {[c[0] if isinstance(c, (list,tuple)) else c for c in candidates]}")
    print()

--- Sample 1: PMC ISO 14530-UP (GF10+MD65),M,FR Ralupol UP 804 7035.00 M/Q ---
  Core concept: molding
  Profile keys: ['entity_name', 'core_concept', 'distinguishing_features', 'key_properties', 'technical_specifications', 'alternative_names', 'classification_sourcees', 'constituent_materials', 'manufacturing_processes', 'applications', 'notes', '_metadata']
  Ground truth: Glass fibre reinforced plastic, polyester resin, hand lay-up {GLO}| market for glass fibre reinforced plastic, polyester resin, hand lay-up | Cut-off, S
  Top 5 candidates: ['Glass fibre reinforced plastic, polyester resin, hand lay-up {GLO}| market for glass fibre reinforced plastic, polyester resin, hand lay-up | Cut-off, S', 'Glass fibre reinforced plastic, polyamide, injection moulded {GLO}| market for glass fibre reinforced plastic, polyamide, injection moulded | Cut-off, S', 'Glass fibre reinforced plastic | 50% PA66 50% GF /RER', 'Glass fibre reinforced plastic | 90% PC 10% GF /GLO', 'Glass fibre reinforced 

## 4. Load Baseline & Evaluate

<details>
<summary>Details</summary>

Load the current `llm_ranking` prompt from the synced experiment, wrap it in a PromptState, and evaluate it via the backend.

</details>

In [8]:
#@title Load baseline prompt
baseline = load_baseline_prompt(svc["exp_data"])
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
print(f"Evaluation data: {len(eval_data)} queries")

Evaluation data: 40 queries


In [9]:
#@title Evaluate baseline prompt
campaign_rounds, baseline_results = await run_baseline_eval(
    baseline, eval_data, campaign_config, GROQ_API_KEY, svc,
)

Baseline eval:   0%|          | 0/40 [00:00<?, ?query/s]


Round    Accuracy   Rolling Avg    Trend
  0        17.5%        17.5%  -
  MISS: Round couper wire 2,5mm²/pre-welded braid  |  Pred: Wire drawing, copper {RER}| wire dr  |  GT: Copper, cathode {GLO}| market for c
  MISS: PA66-GF35 Bergamid A700 G35 H RAL3000/molding  |  Pred: Glass fibre reinforced plastic | 65  |  GT: Glass fibre reinforced plastic | 75
  MISS: Round EN 10277-3-11SMn30+C/turning  |  Pred: Steel removed by turning, average,   |  GT: Steel, low-alloyed {GLO}| market fo
  MISS: 6.8. - ISO 898-1/thread rolling  |  Pred: Hot rolling, steel {RoW}| hot rolli  |  GT: Steel, unalloyed {GLO}| market for 
  MISS: PA66-Bergamid A700 CF RAL6018/molding  |  Pred: Glass fibre reinforced plastic | 75  |  GT: Polyamide (Nylon) 6.6/EU-27


## 4.5 Smart Prompt Search — Sensitivity-Guided Optimization

Measures each axis (prompt fields + pipeline params) one-at-a-time against the baseline,
classifies them by sensitivity, then runs coordinate descent on the axes that matter.
Typically finds the best pipeline params (temperature, sample size) automatically
while deferring nuanced prompt changes to the iterative optimizer (Section 5).

In [10]:
#@title 4.5a � Build diagnostic set (with resume)
variant_library = load_variant_library()
ss = campaign_config.get("smart_search", {})

llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)

_result = await resume_or_build_diagnostic(
    campaign_config, baseline, baseline_results,
    llm_client, llm_model,
    svc["store"], svc["backend_id"], eval_data,
    improvement_areas=improvement_areas,
)
plan_id, search_baseline, diagnostic, diag_summary, cached_profiles = _result

2026-02-25 08:40:36 INFO     [api.services.search.smart_search] Adopting scan data from sibling plan ssplan_21784fea1650 (5 profiles)


[RESUME] Plan ssplan_48678e858bb5: 5 axis profiles available


In [11]:
#@title 4.5a.1 — Historical data audit
prompt_index = build_historical_index(svc["store"], svc["backend_id"])

# Try to synthesize sensitivity from grid data
if not cached_profiles:
    synth = synthesize_sensitivity(
        svc["store"], svc["backend_id"], prompt_index, diagnostic,
    )
    if synth:
        scan_df, axis_profiles = synth
        cached_profiles = axis_profiles
        print("Sensitivity derived from grid data — scan may be skippable.")

2026-02-25 08:40:38 INFO     [api.services.search.coverage] build_prompt_result_index: 111 runs -> 109 unique prompts, 419 total query results


In [12]:
#@title 4.5a.1b — Data inventory
inventory = show_data_inventory(prompt_index, svc["store"], svc["backend_id"])

  DATA INVENTORY  (109 prompts, 419 query results)
  Baselines: 3 plan baseline(s) — 47 queries cached

  Axis                    Prompts  Queries  Distinct values
  persona                      20       20                3
  task_intent                  17       17                2
  thinking_style               22       22                3
  answer_format                15       15                1

  Pipeline parameters (from sensitivity scans):
    max_token_candidates     3 values scanned  sensitivity: 0.000  [skip]
    ranking_sample_size      3 values scanned  sensitivity: 0.000  [skip]
    ranking_temperature      4 values scanned  sensitivity: 0.000  [skip]

  Identified: 36/109 prompts (80/419 queries) via stored plans
  Unmatched:  73 prompts (339 queries)


In [13]:
#@title 4.5a.2 — Coverage advisor
# Knobs: adjust these and re-run to see different strategies
min_queries = 6          # min queries per variant to count as "usable"
axis_requirements = None  # None = require all values; or e.g. {"persona": 2}

coverage = show_scan_coverage(
    search_baseline, variant_library, diagnostic,
    prompt_index,
    pipeline_params=campaign_config.get("pipeline_params"),
    min_queries=min_queries,
    axis_requirements=axis_requirements,
)

  COVERAGE ADVISOR  (min_queries=6)
  Baseline: 6/6 queries cached ✓

  Prompt field axes:
    persona                4 values | 0/4 required  ✗  (0 usable, 4 uncovered)
    task_intent            3 values | 3/3 required  ✓  (3 usable)
    thinking_style         4 values | 4/4 required  ✓  (4 usable)
    answer_format          2 values | 0/2 required  ✗  (0 usable, 2 uncovered)
    problem_description    1 value  | 0/1 required  ✗  (0 usable, 1 uncovered)

  Pipeline params (always need backend):
    ranking_temperature    4 variants × 6 queries = 24 calls
    max_token_candidates   3 variants × 6 queries = 18 calls
    ranking_sample_size    3 variants × 6 queries = 18 calls

  Summary: 42 cached, 102 still needed (42 prompt-field + 60 pipeline-param)
  >> 2/5 prompt field axes covered. Run scan to fill gaps on: persona, answer_format, problem_description, ranking_temperature, max_token_candidates, ranking_sample_size. Tip: lower min_queries or reduce axis_requirements to accept spars

In [14]:
#@title 4.5b — Sensitivity scan
if cached_profiles:
    print(f"[RESUME] Sensitivity scan already complete, "
          f"loaded {len(cached_profiles)} axis profiles")
    scan_df = None  # Not needed for adaptive search
    axis_profiles = cached_profiles
    display_axis_profiles(axis_profiles)
else:
    scan_df, axis_profiles = await sensitivity_scan(
        search_baseline, variant_library, diagnostic, svc.get("backend_client"),
        user_focus=improvement_areas,
        store=svc["store"], backend_id=svc["backend_id"],
        pipeline_params=campaign_config.get("pipeline_params"),
        session_terms=svc.get("session_terms"),
        plan_id=plan_id,
        prompt_result_index=prompt_index,
    )

[RESUME] Sensitivity scan already complete, loaded 5 axis profiles

Rank  Axis                      Type            Card  Range    Budget  
----------------------------------------------------------------------
  1   thinking_style            prompt_field    4     0.500    high    
  2   task_intent               prompt_field    3     0.167    medium  
  3   max_token_candidates      pipeline_param  3     0.000    skip    
  4   ranking_sample_size       pipeline_param  3     0.000    skip    
  5   ranking_temperature       pipeline_param  4     0.000    skip    


In [ ]:
#@title 4.5c — Select best from scan & seed campaign (no backend needed)
best_ps, best_params = select_scan_winner_notebook(
    scan_df, axis_profiles, search_baseline, variant_library,
    pipeline_params=campaign_config.get("pipeline_params"),
    store=svc["store"], backend_id=svc["backend_id"], plan_id=plan_id,
)

if best_params:
    campaign_config["pipeline_params"] = best_params
    print(f"Updated pipeline_params: {best_params}")

campaign_rounds.append({
    "round": "search",
    "label": f"smart_search ({best_ps.changes_description or best_ps.id[:12]})",
    "prompt_state": best_ps,
    "accuracy": campaign_rounds[0]["accuracy"] if campaign_rounds else 0.0,
    "hits": campaign_rounds[0].get("hits", 0) if campaign_rounds else 0,
    "total": campaign_rounds[0].get("total", 0) if campaign_rounds else 0,
    "results": campaign_rounds[0].get("results", []) if campaign_rounds else [],
})
display_progress(campaign_rounds)

## 4.6 Grid Search — Landscape Exploration (Alternative)

<details>
<summary>Skip if you used Smart Search (4.5) above. Expand for brute-force grid sweep.</summary>

**What:** Systematic sweep of the prompt configuration space (Layer 1 fields) using a cartesian product of default axis variations. Maps the accuracy landscape before hill-climbing.

**When to use:** When you want exhaustive coverage of the grid, or when Smart Search results look unreliable and you want independent validation.

**What you get:** Ranked starting points, which dimensions matter most (marginal stats), interaction effects between fields (heatmaps), and LLM-analyzed insights.

**How to read results:**
- **Ranked table** — best combos at the top; use the winner as your campaign seed
- **Marginal stats** — which axis values have the highest mean accuracy across all combos
- **Pairwise heatmaps** — green = good interaction, red = bad; look for synergies and conflicts
- **LLM analysis** — automated pattern recognition across the grid results

</details>

In [ ]:
#@title 4.6 — Grid Campaign Overview (existing plans)
merge_plans = False  # Set True to combine results from multiple plans
grid_overview = show_grid_overview(svc, campaign_config, merge_plans=merge_plans)
merged_grid_df = grid_overview.get("merged_grid_df")

In [ ]:
#@title 4.6a � Build or resume grid plan + load eval data
gs = campaign_config["grid_search"]
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)

(
    grid_plan_id, grid_points, grid_state_lookup,
    grid_axes, layer1_fields, grid_baseline,
) = await resume_or_build_grid(
    campaign_config, baseline, llm_client, llm_model,
    svc["store"], svc["backend_id"],
    improvement_areas=improvement_areas,
)

# Load full eval_data for later optimization rounds
eval_data = load_eval_dataset(
    svc["store"], svc["backend_id"], svc["experiment_id"],
)
if not eval_data:
    raise RuntimeError(
        "No evaluation data found. Generate data first "
        "(e.g. run termnorm_backend.ipynb or another data source)."
    )

print(f"
Grid points: {len(grid_points)}")
print(f"Plan ID: {grid_plan_id}")

In [ ]:
#@title 4.6c — Run grid search
grid_df = await run_grid_search(
    grid_points, grid_state_lookup, eval_data,
    campaign_config["eval_llm"], GROQ_API_KEY,
    plan_id=grid_plan_id,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_client=svc.get("backend_client"),
    session_terms=svc.get("session_terms"),
    pipeline_params=campaign_config.get("pipeline_params"),
    eval_queries_per_point=gs.get("eval_queries_per_point", 1),
    shared_queries=gs.get("shared_queries", False),
    grid_seed=gs.get("seed", 42),
)

In [ ]:
#@title 4.6d — Display grid results
_display_df = merged_grid_df if merged_grid_df is not None else grid_df
display_grid_results(_display_df, grid_axes, top_k=gs.get("top_k", 5))

In [ ]:
#@title 4.6e � LLM analysis of grid results
_analysis_df = merged_grid_df if merged_grid_df is not None else grid_df
llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)
grid_analysis = await analyze_grid_results(
    _analysis_df, grid_axes, llm_client, model=llm_model,
)

In [ ]:
#@title 4.6f — Select grid winner and seed campaign
grid_winner = select_and_seed_grid_winner(
    grid_df, merged_grid_df, grid_state_lookup,
    grid_overview.get("plan_dfs", {}), svc, campaign_rounds,
)

## 5. Run Optimization

<details>
<summary>Details</summary>

Two modes:
- **Semi-automatic** (recommended): runs multiple rounds with patience-based auto-stop
- **Manual**: run one round at a time for full HITL control

Both modes subsample `eval_data` to `queries_per_eval` queries per step.

</details>

In [ ]:
#@title Run optimization (feedback cycle — M3 nodes)
campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_url=svc["backend_client"].base_url,
    pipeline_params=campaign_config.get("pipeline_params"),
)

In [ ]:
#@title Run optimization round (manual)
round_entry = await run_manual_round(
    campaign_rounds, eval_data, campaign_config, GROQ_API_KEY, svc,
)

## 6. LLM Suggestion for Next Round (HITL)

<details>
<summary>Details</summary>

After each round, the LLM analyzes failures and suggests:
1. Failure pattern analysis
2. Parameter change suggestions
3. Prompt phrase fragments to adopt
4. Suggested next `campaign_config`

**Review the suggestions, edit the config cell (Section 2), then re-run Sections 5-6.**

</details>

In [ ]:
#@title Generate LLM suggestions for next round
llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)
suggestions = await generate_suggestions(
    campaign_rounds, eval_data, campaign_config,
    llm_client, model=llm_model,
)
display_suggestions(suggestions, len(campaign_rounds))
print(f"
--- SUGGESTED CONFIG (copy to Section 2) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

## 7. Campaign Summary

<details>
<summary>Details</summary>

Compare all rounds, track per-query flips, display the PromptState lineage chain, and save the winner.

</details>

In [ ]:
#@title Campaign comparison table
rows = []
for rd in campaign_rounds:
    rows.append({
        "round": rd["round"],
        "label": rd["label"][:40],
        "hit@1": rd["hits"],
        "total": rd["total"],
        "accuracy": f"{rd['accuracy']:.1%}",
        "prompt_id": rd["prompt_state"].id[:12],
    })

print(f"CAMPAIGN SUMMARY ({len(campaign_rounds)} rounds)")
print(f"{'='*70}")
display(pd.DataFrame(rows))

In [ ]:
#@title Per-query flip tracking (baseline vs final)
if len(campaign_rounds) >= 2:
    base_r = campaign_rounds[0]["results"]
    final_r = campaign_rounds[-1]["results"]

    flips = []
    for br, fr in zip(base_r, final_r):
        b_hit = br["hit"]
        f_hit = fr["hit"]
        if b_hit != f_hit:
            flips.append({
                "query": br["query"][:50],
                "flip": "MISS->HIT" if f_hit else "HIT->MISS",
                "base_pred": br["predicted"][:35],
                "final_pred": fr["predicted"][:35],
                "ground_truth": br["ground_truth"][:35],
            })

    gained = sum(1 for f in flips if f["flip"] == "MISS->HIT")
    lost = sum(1 for f in flips if f["flip"] == "HIT->MISS")

    print(f"FLIP TRACKING (baseline -> round {campaign_rounds[-1]['round']})")
    print(f"  Queries gained (MISS->HIT): {gained}")
    print(f"  Queries lost (HIT->MISS):   {lost}")
    print(f"  Net change:                 {gained - lost:+d}")
    print()
    if flips:
        display(pd.DataFrame(flips))
else:
    print("Need at least 2 rounds for flip tracking.")

In [ ]:
#@title PromptState lineage chain
print("LINEAGE CHAIN")
print("="*50)
for i, rd in enumerate(campaign_rounds):
    ps = rd["prompt_state"]
    parent = ps.parent_id[:12] if ps.parent_id else "root"
    arrow = "  " if i == 0 else "  -> "
    print(f"{arrow}[{ps.id[:12]}] Round {rd['round']}: {rd['label'][:40]} ({rd['accuracy']:.1%})")
    if ps.parent_id:
        print(f"       parent: {parent}  |  changes: {ps.changes_description or 'none'}")

In [ ]:
#@title Save winner
save_campaign_winner(campaign_rounds, campaign_config, svc["store"], svc["backend_id"])